# 07 — Vietnamese Symptom Extraction: Rule-based, Semantic và Hybrid

## 1. Câu hỏi nghiên cứu

Semantic retrieval có tăng recall cho cách nói ngoài từ điển mà không tăng nhầm phủ định, chủ thể hoặc hư cấu? Hybrid có giảm abstention so với quy tắc hiện tại? Đây là nghiên cứu NLP độc lập, **không phải Experiment 5 của lựa chọn disease classifier**. Không load LR, fit model bệnh, mở locked disease test hoặc gọi API LLM.

Giả thuyết chưa được xác nhận. Kết quả hiện có chỉ là baseline thực trên câu synthetic; Semantic/Hybrid **NOT RUN** do chưa có embedding snapshot/dependency đủ điều kiện. Không tự bật phương án NLP lên website.

Viewer chạy được trên Colab hoặc Python local chỉ với chính notebook này: snapshot bảng/hình đã xác minh được nhúng dưới dạng dữ liệu có SHA-256, không phụ thuộc cwd, Drive hoặc ổ Windows. Có thể chạy các viewer cells với hai gate False; không thực thi benchmark mới. Chạy nghiên cứu mới vẫn cần source đầy đủ và interpreter research riêng; không tạo bootstrap hoặc tự download dependencies/weights.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, sys
from IPython.display import display, Markdown, Image

# Reviewed saved evidence embedded as data, not executable source or new results.
# This viewer needs no Windows path, Drive mount, checkpoint, or API credentials.
VIEWER_ASSETS = {'benchmark.json': {'base64': '...', 'sha256': '51c4dc5ba98902f1b008f4654f6f866b9fc9d298a5d8f524ed294661108f293d', 'source_sha256': '51c4dc5ba98902f1b008f4654f6f866b9fc9d298a5d8f524ed294661108f293d'}, 'coverage_inventory.json': {'base64': '...', 'sha256': 'd8c666e2f5a8f6d92403d61c920b10c973441d638334709a2e6c762bb6364a98', 'source_sha256': '6f77155d8651512fd97211f5a65abec8e4a4d1c4461ec787cbd3f24600a08c5b'}, 'context_after_fix.json': {'base64': '...', 'sha256': '56eba2a32e152542d7437add4091896356c911e87fb37d4a535491e8fc6ef17b', 'source_sha256': '95a157d6f3da6c18358851d450406187f194045a3df20095256d60df3e575c2c'}, 'rule_metrics.png': {'base64': '...', 'sha256': '0db89dda2905438593e2c28325aac94f21d888d7e2aeead04860b1affdb99cee', 'source_sha256': '0db89dda2905438593e2c28325aac94f21d888d7e2aeead04860b1affdb99cee'}, 'context_errors.png': {'base64': '...', 'sha256': '76f80396e0fa1656e15ba1730dcfe70d4d8fde255d3b8bf6913ae59948ad36a9', 'source_sha256': '76f80396e0fa1656e15ba1730dcfe70d4d8fde255d3b8bf6913ae59948ad36a9'}, 'activation_coverage.png': {'base64': '...', 'sha256': '86b8f5e261eb1bb7f8cdf415723d84c73473eb4539563d1314a72a879b3e1dfd', 'source_sha256': '86b8f5e261eb1bb7f8cdf415723d84c73473eb4539563d1314a72a879b3e1dfd'}, 'rule_latency.png': {'base64': '...', 'sha256': '856b3030fb3f1817e24b92db5e81aa0486afd4368945cc97b3031c844cfb7d3c', 'source_sha256': '856b3030fb3f1817e24b92db5e81aa0486afd4368945cc97b3031c844cfb7d3c'}, 'nlp_research.json': {'base64': 'ew0KICAicHJvdG9jb2wiOiAidmlldG5hbWVzZS1ubHAtcmVzZWFyY2gtdjEiLA0KICAiY3JlYXRlZCI6ICIyMDI2LTExLTExIiwNCiAgInByb2R1Y3Rpb25fYWN0aXZhdGlvbiI6IGZhbHNlLA0KICAiY2F0YWxvZ19mZWF0dXJlcyI6IDM3NSwNCiAgInByb2R1Y3Rpb25fZmVhdHVyZXNfYmVmb3JlIjogMTM1LA0KICAiZGV2ZWxvcG1lbnQiOiAiZGF0YS9ldmFsdWF0aW9uL3ZpZXRuYW1lc2Vfc3ltcHRvbV9zZW1hbnRpY19jYXNlcy5qc29ubCIsDQogICJyZXNlcnZlZCI6ICJkYXRhL2V2YWx1YXRpb24vdmlldG5hbWVzZV9ubHBfcmVzZXJ2ZWRfdjEuanNvbmwiLA0KICAicmVzZXJ2ZWRfcm93cyI6IDMyLA0KICAicmVzZXJ2ZWRfc2hhMjU2IjogIjVjZDhiZDUyZTAyYTUwNmVlYjIxYTNkNTRlYzI3YjM2MzQ4NWFlMWQ4ZTQyYzgxNGQ5MGE2MWM5ZTE4N2Y4MjQiLA0KICAic3BsaXRfdW5pdCI6ICJ3aG9sZSBpbmRlcGVuZGVudGx5IHdyaXR0ZW4gdXR0ZXJhbmNlOyBleGFjdCBub3JtYWxpemVkIHRleHQgbXVzdCBub3QgY3Jvc3Mgc3BsaXRzIiwNCiAgInNwbGl0X2xpbWl0YXRpb25zIjogIlNtYWxsIHN5bnRoZXRpYyByZXNlcnZlZCBzZXQsIHNhbWUgYXV0aG9ycyBhbmQgc3ltcHRvbSBjb25jZXB0cyBhcyBkZXZlbG9wbWVudDsgcGVuZGluZyBodW1hbiByZXZpZXcsIG5vdCBpbmRlcGVuZGVudCBjbGluaWNhbCBvciBwYXJhcGhyYXNlLWZhbWlseSBnZW5lcmFsaXphdGlvbiIsDQogICJhY3RpdmF0aW9uX3Byb2JlIjogInByb3Bvc2VkIGFsaWFzZXMgaW4gcGVyLWZlYXR1cmUgdGVtcG9yYXJ5IGRpY3Rpb25hcmllczsgZGV2ZWxvcG1lbnQgdGVtcGxhdGVzIG9ubHksIG5ldmVyIGVkaXRzIHByb2R1Y3Rpb24iLA0KICAic2VtYW50aWNfcG9saWN5IjogImxvY2FsLW9ubHkgcGlubmVkIGVtYmVkZGluZzsgdG9wMyBjYW5kaWRhdGVzIHBlciBjbGF1c2UsIGNvbmZpcm1hdGlvbiByZXF1aXJlZCwgbm8gYXV0b21hdGljIHNpbWlsYXJpdHkgdGhyZXNob2xkIiwNCiAgInR1bmluZ19wb2xpY3kiOiAibm8gdGhyZXNob2xkIHNlbGVjdGVkOyBmdXR1cmUgYXV0by1zZWxlY3Rpb24gcmVxdWlyZXMgbmV3IHByZXJlZ2lzdHJhdGlvbiBhbmQgZGV2ZWxvcG1lbnQtb25seSBzZWxlY3Rpb24gYmVmb3JlIG9wZW5pbmcgYSBmcmVzaCBOTFAgZXZhbHVhdGlvbiBzZXQiLA0KICAiZGlzZWFzZV90ZXN0X2FjY2VzcyI6IGZhbHNlLA0KICAibGl2ZV9hcGlfY2FsbHMiOiBmYWxzZQ0KfQ0K', 'sha256': '17396dd6f5575b5b1641aa36ec53ac140444b2114659c7a1bd8eb8bff59b672b', 'source_sha256': '17396dd6f5575b5b1641aa36ec53ac140444b2114659c7a1bd8eb8bff59b672b'}}

def snapshot_bytes(name):
    item = VIEWER_ASSETS[name]
    raw = base64.b64decode(item['base64'], validate=True)
    if hashlib.sha256(raw).hexdigest() != item['sha256']:
        raise ValueError('Saved viewer snapshot checksum mismatch: ' + name)
    return raw

def snapshot_json(name):
    return json.loads(snapshot_bytes(name).decode('utf-8'))

def find_research_root():
    candidates = []
    if os.environ.get('NLP_PROJECT_ROOT'):
        candidates.append(Path(os.environ['NLP_PROJECT_ROOT']).expanduser())
    runtime = globals().get('RUNTIME')
    if runtime is not None and getattr(runtime, 'paths', None) is not None:
        candidates.append(Path(runtime.paths.project_root))
    candidates.extend((Path.cwd(), *Path.cwd().parents, Path('/content/disease-prediction-ml-runtime')))
    required = ('configs/nlp_research.json', 'scripts/nlp_symptom_research.py',
                'data/evaluation/vietnamese_nlp_reserved_v1.jsonl')
    return next((p.resolve() for p in candidates if all((p / n).is_file() for n in required)), None)

ROOT = find_research_root()
REPORT = ROOT / 'reports/nlp_research_v1' if ROOT is not None else None
protocol = snapshot_json('nlp_research.json')
saved = snapshot_json('benchmark.json')
inventory = snapshot_json('coverage_inventory.json')
after = snapshot_json('context_after_fix.json')
print('Viewer: verified saved snapshot, not a new benchmark run.')
print('Research source:', ROOT if ROOT is not None else 'NOT AVAILABLE; viewer still works on Colab/local.')
print(saved['evaluation_status'])
print('Semantic:', saved['semantic_environment']['status'], '|', saved['semantic_environment']['reason'])


## 2. Benchmark và provenance

86 fixtures tiếng Việt lịch sử chỉ dùng development; 63 câu có expected states đầy đủ, 23 câu chỉ gán nhãn từng phần/forbidden features. Không coi phần chưa gán nhãn là negative. Thêm 32 câu reserved synthetic v1, pending_human_review, không dùng trong alias probes/threshold selection; exact normalized text không giao nhau giữa hai tập. Cùng tác giả và khái niệm triệu chứng vẫn tạo bias: đây không phải bộ bệnh nhân độc lập hoặc clinical validation.

Mỗi ca mới ghi canonical states, subject=self, ambiguity, evidence offsets khi có, provenance và flags tách evaluation khỏi development. Các ca other-subject/hư cấu kỳ vọng không có state cho người dùng. Fixture lịch sử không có span gold được ghi nhận giới hạn, không bịa span annotation.

**Reserved đã được quan sát trong lần baseline đầu**, phát hiện hai false positive hư cấu. Giữ nguyên kết quả trước sửa; không chạy lại để gọi đó là cải thiện độc lập. Cần bộ NLP evaluation mới được người duyệt trước kết luận tổng quát hoặc chọn threshold. Đây không phải locked final disease test.


In [ ]:
display(protocol)
for split, methods in saved['results'].items():
    m = methods['rule_based']['metrics']
    print(split, '| cases:', m['cases'], '| fully annotated:', m['fully_annotated_cases'], '| partial:', m['partial_label_cases'])
display(saved['inputs_sha256'])
display(saved['baseline_source_sha256'])


## 3. Existing Rule-based baseline và inventory 375 feature

Helper gọi production `OfflineConversation.handle` → `OfflineExtractor` → canonical validator thật; không viết bản rule đơn giản để thay baseline, không cung cấp prediction service. Chỉ đọc trạng thái/evidence, kiểm span slices và giải phóng session từng ca.

Inventory giữ schema order, display_vi, mô tả phân biệt, aliases, nguồn/chưa xác minh, collision, forbidden contexts và trạng thái production. DRAFT = dự thảo; LEXICALLY_REVIEWED = rà từ vựng nội bộ, không thẩm định y khoa; CONTEXT_TESTED = chỉ các template đã thử; PRODUCTION_ACTIVE = đang được runtime dùng; REVIEW_REQUIRED = còn blocker. Các mức này không tương đương clinical review.

375/375 đã có tên và ghi chú trước nhiệm vụ; không gọi đó là 375/375 production. Probes mỗi feature dùng dictionary tạm, không sửa synonyms. Kết quả sau sửa guard lưu riêng; các feature còn semantic overlap/decision HOLD/CLARIFY/REVISE không được kích hoạt. Việc qua vài template cũng chưa thay cho batch release regression.


In [ ]:
display(inventory['summary'])
display(after['summary'])
display([dict(feature=r['canonical_feature'], display_vi=r['display_vi'], lifecycle=r['lifecycle'], blockers=r['activation_blockers'])
         for r in after['features'][:8]])
print('Catalog đầy đủ:', len(after['features']), 'features trong snapshot; CSV source:', REPORT / 'canonical_375.csv' if REPORT else 'reports/nlp_research_v1/canonical_375.csv trong repository')


## 4. Semantic matching — chỉ nghiên cứu, chưa chạy

Ứng viên: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, revision `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`, Apache-2.0; publisher liệt kê tiếng Việt. [Model card](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), [metadata publisher](https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). Khoảng 118M parameters, embedding 384 chiều; cần tự đo RAM/latency trên máy nghiên cứu, không coi là model y khoa.

Chưa tải weights. Dùng interpreter research riêng có `sentence-transformers` và PyTorch CPU tương thích; không cài/chỉnh dependencies interpreter API để chạy notebook này. Tải thủ công từ publisher với revision trên vào cache ngoài repo, theo license. Snapshot phải có `research_snapshot.json` gồm `model_id`, `revision`, `files` (relative path → SHA256 của mọi file khác manifest), có safetensors; không symlink/file ngoài snapshot. Không tạo manifest cho weights không rõ provenance. Helper xác minh inventory/checksum trước import/load, dùng `local_files_only=True`, `trust_remote_code=False`; không download từ notebook.

Chia theo clause/span giữ offsets gốc, truy hồi Top 3 ứng viên từ 375 descriptions. Không embedding toàn hội thoại để ép chọn một triệu chứng. Cosine similarity chỉ xếp hạng ứng viên, không phải xác suất bệnh. Không có threshold được chọn/giả lập; tất cả ứng viên semantic chờ người dùng xác nhận, không tự thành present.


In [ ]:
RUN_RESEARCH = False
RUN_SEMANTIC = False
EMBEDDING_SNAPSHOT = os.environ.get('NLP_EMBEDDING_SNAPSHOT')
if RUN_RESEARCH:
    if ROOT is None:
        raise RuntimeError('Chưa có research source. Mở kernel local trong Git checkout hoặc chuẩn bị source đầy đủ qua shared runtime; viewer không cần bước này. Không upload từng module hay bỏ checksum.')
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from datetime import datetime, timezone
    from scripts.nlp_symptom_research import run
    output = ROOT / 'reports' / ('nlp_research_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    # Reserved v1 is already observed: reruns are reproducibility checks, not a new held-out claim.
    if RUN_SEMANTIC and not EMBEDDING_SNAPSHOT:
        raise RuntimeError('Configure a verified local embedding snapshot first')
    new_inventory, research = run(output, ROOT, EMBEDDING_SNAPSHOT if RUN_SEMANTIC else None)
    print('New research output:', output)
else:
    print('Viewer only; no extraction, embedding load, network or disease prediction.')


## 5. Hybrid matching

Hybrid giữ nguyên validated rule states; retrieval chỉ bổ sung ứng viên chưa có trong rule state và yêu cầu confirmation. Semantic không override denied/uncertain hoặc other-subject/fiction guard. Nghiên cứu dùng lại `evidence_context` để lọc evidence theo subject/time/assertion; điều đó chỉ chứng minh context, không chứng minh alias đồng nghĩa.

Không giả lập câu trả lời “có” để tăng recall. Khi chưa có người xác nhận, semantic-only abstain; hybrid chỉ có committed states từ rules. Benchmark tương lai phải phân biệt candidate recall@K với recall của states đã xác nhận, và đánh giá thêm user confirmation thật. Không có tích hợp semantic production trong nhiệm vụ này.


## 6. Evaluation và mẫu số

Precision/Recall/Micro F1 tính trên cặp `(canonical feature, assertion status)` của câu fully annotated; zero_division=0. Macro F1 trung bình trên union cặp gold/pred đã quan sát, báo denominator; không phải macro 587 disease labels hay coverage đủ 375 triệu chứng. Exact-set match so toàn bộ states của câu fully annotated. Partial cases chỉ kiểm forbidden-feature violations, không đưa vào P/R/F1.

False positive breakdown ghi theo loại context; FN ghi riêng. Mapping coverage là số feature khác nhau được nhận diện / 375, không phải activation coverage. Confirmation rate là tỷ lệ câu cần clarification, không phải xác nhận đúng hoặc thành công hội thoại. Latency p50/p95 đo một lượt trên máy này, không warm-up/multi-user benchmark hoặc cam kết SLA.


In [ ]:
import pandas as pd
rows = []
for split, methods in saved['results'].items():
    for method, result in methods.items():
        rows.append({'split': split, 'method': method, 'status': result['status'], **result.get('metrics', {})})
display(pd.DataFrame(rows).drop(columns=['false_positive_contexts', 'macro_definition', 'confirmation_definition'], errors='ignore'))
display({split: methods['rule_based']['metrics']['false_positive_contexts'] for split, methods in saved['results'].items()})


## 7. Hình từ kết quả đã đo

Chỉ vẽ Rule-based initial baseline và activation coverage thực. Không vẽ Semantic/Hybrid bars khi NOT RUN. Nguồn: `reports/nlp_research_v1/benchmark.json` và coverage inventory, checksum trong figure manifest. Biểu đồ context là trước sửa guard, không mô tả runtime sau sửa.


In [ ]:
for name in ('rule_metrics.png', 'context_errors.png', 'activation_coverage.png', 'rule_latency.png'):
    display(Image(data=snapshot_bytes(name), format='png'))


## 8. Kết luận và giới hạn

Rule-based initial baseline đạt kết quả tốt trên fixtures development đã quen thuộc nhưng còn false positive hư cấu ở reserved synthetic set. Guard được sửa theo câu chứa evidence, có regression riêng; không đổi điểm baseline cũ hoặc gọi đó là semantic improvement. Không kết luận Semantic/Hybrid tốt hơn khi chưa load model và chưa đánh giá.

Giữ nguyên 135 production features cho đến khi aliases vượt context, semantic-overlap và batch release checks; inventory đầy đủ không tự cho phép activation. Các giả định của nhãn tổng hợp, collision không dấu, typo, phủ định phức tạp, đồng nghĩa và cách nói ngoài catalog vẫn cần người duyệt. 32 câu synthetic không đủ chứng minh chất lượng extraction ngoài đời.

Xem [README](../README.md), [NLP contract](../docs/phase5_llm_integration_design.md), [development log](../docs/DEVELOPMENT_LOG.md). Notebook 01–06, model LR tuned, VT318, schema375/class587 và scientific results không được sửa bởi nghiên cứu này.
